# 05 — Evaluation Analysis

Compute retrieval hit@1, hit@3, hit@5 from `data/eval/eval_questions.csv`. Optionally run answer evaluation (requires indexing + `OPENAI_API_KEY`). Paths use project config.

**Prerequisites:** For retrieval eval: run indexing pipeline. For answer eval: also set `OPENAI_API_KEY`.

**Interpretation:** hit@1 = strict (correct doc must be rank 1); hit@3 and hit@5 are more lenient. Higher is better; compare across runs to see impact of chunking or model changes.

In [ ]:
from src.evaluation.evaluate_retrieval import evaluate_retrieval

r = evaluate_retrieval()
if "error" in r:
    print("Error:", r["error"])
else:
    n = r.get("n_questions", 0)
    print("N questions:", n)
    print("Hit counts (number of questions with expected doc in top-k):", r.get("hit_counts", {}))
    print("Hit rates (fraction):")
    for k, v in (r.get("hit_rates") or {}).items():
        print(f"  {k}: {v:.2%}" if v is not None else f"  {k}: {v}")

In [ ]:
# Optional: run QA for all eval questions and save to artifacts/eval_results
# use_llm=False avoids OPENAI_API_KEY (retrieval-only); use_llm=True for full answers
from src.config import EVAL_RESULTS_DIR
from src.evaluation.evaluate_answers import evaluate_answers_batch

answer_df = evaluate_answers_batch(use_llm=False)
if not answer_df.empty:
    print("Answer eval preview:")
    display(answer_df.head())
    print(f"Full results saved to: {EVAL_RESULTS_DIR / 'answer_eval.csv'}")
else:
    print("No eval questions found or eval file missing.")

In [ ]:
# Per-question breakdown: which questions have expected doc in top-1 / top-3 / top-5
import pandas as pd
from src.config import EVAL_QUESTIONS_PATH
from src.retrieval.retrieve import retrieve_top_k

def expected_in_top(doc_names, expected, k):
    top = doc_names[:k]
    exp = (expected or "").strip()
    for n in top:
        if exp == n or exp in n or n in exp:
            return True
    return False

eval_df = pd.read_csv(EVAL_QUESTIONS_PATH)
rows = []
for _, row in eval_df.iterrows():
    q, exp = row["question"], row.get("expected_document", "")
    if pd.isna(exp) or not str(exp).strip():
        continue
    results = retrieve_top_k(q, top_k=5)
    doc_names = [r.get("document_name", "") for r in results]
    rows.append({
        "question": (q[:48] + "…") if len(q) > 50 else q,
        "expected_document": exp,
        "hit@1": expected_in_top(doc_names, exp, 1),
        "hit@3": expected_in_top(doc_names, exp, 3),
        "hit@5": expected_in_top(doc_names, exp, 5),
    })
breakdown = pd.DataFrame(rows)
display(breakdown)